<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_27_concurrency_intro/note_lesson_27_concurrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 27 — Потоки, процеси, asyncio: ранковий звіт диспетчерської

Диспетчерська «Смачно + Таксі» щоранку опитує 5 ресторанів (кожна відповідь — 0,5 с очікування мережі) і рахує важку статистику по 4 районах. Сьогодні пришвидшуємо обидві задачі — і вчимося обирати інструмент.

| Крок | Що робимо |
|---|---|
| 0 | задачі, що **чекають**, і задачі, що **рахують** |
| 1 | потоки: `ThreadPoolExecutor` (вправа 1) |
| 2 | стан гонитви й `Lock` (вправи 2–3) |
| 3 | GIL і процеси: `ProcessPoolExecutor` (вправа 4) |
| 4 | `asyncio`: `async def`, `await`, `gather`, тайм-аут (вправи 5–6) |
| 5 | знайди помилку (вправа 7) |

**Як працювати:** виконуй клітинки **зверху вниз**. Перед **🔮 Прогнозом** спершу відповідай сам. Час у тебе буде трохи іншим, ніж у тексті, — важливе співвідношення. Теорія й покрокові схеми — у книзі: [Урок 27](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).

---
## 0. Чекати чи рахувати

Функції для **процесів** мають жити у **файлі**: на Windows і macOS процес-працівник імпортує їх заново, а клітинку ноутбука імпортувати не можна. Тому запишемо їх у модуль `district_stats.py` — клітинка з `%%writefile` створює файл поруч з ноутбуком (і в Jupyter, і в Colab).

In [ ]:
%%writefile district_stats.py
"""Функції, які запускаємо в процесах: мають бути у файлі, щоб процеси могли їх імпортувати."""

orders = []


def count_primes(limit):
    """Кількість простих чисел, менших за limit. Навмисно повільно: перебір дільників."""
    count = 0
    for n in range(2, limit):
        for d in range(2, int(n ** 0.5) + 1):
            if n % d == 0:
                break
        else:
            count += 1
    return count


def add_order(name):
    orders.append(name)      # додає у СВОЮ копію списку
    return len(orders)

In [ ]:
import asyncio
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import district_stats

RESTAURANTS = ["Борщ і Ко", "Піца Поділ", "Суші Оболонь", "Вареники 24/7", "Шаурма Центр"]
DISTRICTS = [150_000] * 4


def check_restaurant(name):
    time.sleep(0.5)            # чекаємо відповідь мережі
    return f"{name}: відкрито"


def timed(label, work):
    start = time.perf_counter()
    result = work()
    print(f"{label}: {time.perf_counter() - start:.1f} с")
    return result


timed("5 ресторанів послідовно", lambda: [check_restaurant(n) for n in RESTAURANTS])
timed("4 райони послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])

Ресторани — задача, що **чекає** (IO-bound): процесор простоює, поки йде відповідь. Райони — задача, що **рахує** (CPU-bound): процесор зайнятий на 100 %.

### Вправа 0. Чекає чи рахує?

Для кожної задачі запиши `"IO"` (чекає) або `"CPU"` (рахує).

In [ ]:
kinds = {
    # YOUR CODE HERE
    # BEGIN SOLUTION
    "завантажити 100 сторінок меню з сайтів ресторанів": "IO",
    "стиснути 500 фото страв": "CPU",
    "прочитати 20 файлів звітів з диска": "IO",
    "порахувати найкоротші маршрути для 10 000 поїздок": "CPU",
    "надіслати 1000 SMS через API": "IO",
    # END SOLUTION
}

assert kinds["завантажити 100 сторінок меню з сайтів ресторанів"] == "IO"
assert kinds["стиснути 500 фото страв"] == "CPU"
assert kinds["прочитати 20 файлів звітів з диска"] == "IO"
assert kinds["порахувати найкоротші маршрути для 10 000 поїздок"] == "CPU"
assert kinds["надіслати 1000 SMS через API"] == "IO"
print("✅ Вправа 0 пройдена")

---
## 1. Потоки: чекати одночасно

**🔮 Прогноз:** скільки займе опитування 5 ресторанів через `ThreadPoolExecutor`? У якому порядку будуть результати?

<details>
<summary>Відповідь</summary>

Близько 0,5 с: п'ять очікувань ідуть одночасно. `pool.map` повертає результати **в порядку вхідного списку**, хоч би в якому порядку закінчились потоки.

</details>

In [ ]:
with ThreadPoolExecutor() as pool:
    results = timed("5 ресторанів у потоках", lambda: list(pool.map(check_restaurant, RESTAURANTS)))
print(results)

### Вправа 1. `check_all(names)`

Поверни список статусів для всіх ресторанів, опитуючи їх у потоках. Порядок — як у `names`.

In [ ]:
def check_all(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ThreadPoolExecutor() as pool:
        return list(pool.map(check_restaurant, names))
    # END SOLUTION


start = time.perf_counter()
statuses = check_all(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[0], f"| {elapsed:.1f} с")
assert statuses == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5, "Схоже, ресторани опитуються по черзі"
print("✅ Вправа 1 пройдена")

---
## 2. Стан гонитви

Чотири каси одночасно зараховують по 1000 оплат по 1 грн. «Прочитати» і «записати» — два кроки, а між ними інший потік встигає своє. `time.sleep(0)` лише віддає чергу іншому потоку, щоб проблема проявилася наочно.

**🔮 Прогноз:** скільки буде на рахунку після роботи чотирьох кас?

<details>
<summary>Відповідь</summary>

Значно менше за 4000 — щоразу інше число (у наших 30 запусках — від 1000 до 1336). Каси читають старе значення й записують поверх чужих оплат.

</details>

In [ ]:
balance = 0


def pay(times):
    global balance
    for _ in range(times):
        current = balance        # 1. прочитати
        time.sleep(0)            # каса на мить відволіклася
        balance = current + 1    # 2. записати


threads = [threading.Thread(target=pay, args=(1000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("Очікували 4000, на рахунку:", balance)

### Вправа 2. Виправ через `Lock`

Допиши `pay_safe`: блок «прочитати — записати» має виконуватись під `with lock:`.

In [ ]:
balance = 0
lock = threading.Lock()


def pay_safe(times):
    global balance
    for _ in range(times):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        with lock:
            current = balance
            time.sleep(0)
            balance = current + 1
        # END SOLUTION


threads = [threading.Thread(target=pay_safe, args=(1000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("На рахунку:", balance)
assert balance == 4000
print("✅ Вправа 2 пройдена")

### Вправа 3. Без спільних змінних

Кращий шлях — не ділити змінну взагалі. Нехай кожна каса **повертає**, скільки оплат прийняла, а головна програма підсумовує результати `pool.map`. Жодного `global`, жодного `Lock`.

In [ ]:
def count_payments(times):
    """Каса приймає times оплат по 1 грн і повертає свою суму."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    total = 0
    for _ in range(times):
        time.sleep(0)
        total += 1
    return total
    # END SOLUTION


def total_payments(cashboxes):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ThreadPoolExecutor() as pool:
        return sum(pool.map(count_payments, cashboxes))
    # END SOLUTION


print(total_payments([1000, 1000, 1000, 1000]))
assert total_payments([1000, 1000, 1000, 1000]) == 4000
assert total_payments([5, 7]) == 12
assert total_payments([]) == 0
print("✅ Вправа 3 пройдена")

---
## 3. GIL і процеси

**🔮 Прогноз:** чи пришвидшать 4 потоки підрахунок `count_primes` для 4 районів? А 4 процеси?

<details>
<summary>Відповідь</summary>

Потоки — ні: через **GIL** байт-код у кожен момент виконує лише один потік, вони чергуються. Процеси — так: у кожного свій інтерпретатор і свій GIL, вони рахують на різних ядрах. На нашій 4-ядерній машині: послідовно ≈0,7 с, потоки ≈0,7–0,8 с, процеси ≈0,2 с.

</details>

In [ ]:
timed("послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])
with ThreadPoolExecutor(max_workers=4) as pool:
    timed("4 потоки  ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))
with ProcessPoolExecutor(max_workers=4) as pool:
    timed("4 процеси ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))

У ноутбуці немає `if __name__ == "__main__":` — він і так головна програма. У `.py`-файлі код, що створює процеси, **обов'язково** стоїть під цією перевіркою (розділ «Процеси» в книзі).

Процеси не діляться пам'яттю — перевір:

In [ ]:
district_stats.orders.clear()
with ProcessPoolExecutor(max_workers=1) as pool:
    print(list(pool.map(district_stats.add_order, ["А", "Б", "В"])))
print("orders у ноутбуці:", district_stats.orders)

### Вправа 4. `district_report(limits)`

Поверни список результатів `district_stats.count_primes` для кожного ліміту, рахуючи в процесах. Порядок — як у `limits`.

In [ ]:
def district_report(limits):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ProcessPoolExecutor() as pool:
        return list(pool.map(district_stats.count_primes, limits))
    # END SOLUTION


report = district_report([10, 100, 1000, 150_000])
print(report)
assert report == [4, 25, 168, 13848]
print("✅ Вправа 4 пройдена")

---
## 4. asyncio

`async def` створює **корутину**: виклик не виконує тіло, а повертає об'єкт — як генераторна функція (урок 10). Запускає корутини **цикл подій**; паузу ставить `await`.

У Jupyter і Colab цикл подій уже працює, тому **замість `asyncio.run(main())` пиши `await main()`** прямо в клітинці. У `.py`-файлі — навпаки, `asyncio.run(main())`.

In [ ]:
async def check_restaurant_async(name):
    await asyncio.sleep(0.5)          # чекаємо, але не блокуємо інших
    return f"{name}: відкрито"


coro = check_restaurant_async("Борщ і Ко")
print(type(coro).__name__)
print(await coro)                     # у файлі: asyncio.run(...)

**🔮 Прогноз:** що надрукує `kitchen()` і в якому порядку?

```python
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))
```

<details>
<summary>Відповідь</summary>

`борщ: почав`, `салат: почав`, `салат: закінчив`, `борщ: закінчив`. Борщ доходить до `await` і стає на паузу — цикл подій запускає салат. Салат чекає менше, тому закінчує першим.

</details>

In [ ]:
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))


await kitchen()

### Вправа 5. `check_all_async(names)`

Корутина: опитати всі ресторани одночасно через `asyncio.gather` і повернути список статусів у порядку `names`.

In [ ]:
async def check_all_async(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return await asyncio.gather(*(check_restaurant_async(n) for n in names))
    # END SOLUTION


start = time.perf_counter()
statuses = await check_all_async(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[-1], f"| {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5
print("✅ Вправа 5 пройдена")

### Вправа 6. Тайм-аут

Ресторан, що не відповів за `timeout` секунд, отримує статус `"<назва>: не відповідає"`. Використай `asyncio.wait_for(корутина, timeout=...)` і `except asyncio.TimeoutError` (на Python 3.10 це не той самий клас, що вбудований `TimeoutError`; з 3.11 — той самий).

In [ ]:
async def check_with_delay(name, delay):
    await asyncio.sleep(delay)
    return f"{name}: відкрито"


async def safe_check(name, delay, timeout=1.0):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        return await asyncio.wait_for(check_with_delay(name, delay), timeout=timeout)
    except asyncio.TimeoutError:
        return f"{name}: не відповідає"
    # END SOLUTION


delays = {"Борщ і Ко": 3.0, "Піца Поділ": 0.5, "Суші Оболонь": 0.5, "Вареники 24/7": 0.5, "Шаурма Центр": 0.5}
start = time.perf_counter()
lines = await asyncio.gather(*(safe_check(n, d) for n, d in delays.items()))
elapsed = time.perf_counter() - start
for line in lines:
    print(line)
print(f"{elapsed:.1f} с")
assert lines[0] == "Борщ і Ко: не відповідає"
assert all(line.endswith("відкрито") for line in lines[1:])
assert elapsed < 2.0
print("✅ Вправа 6 пройдена")

---
## 5. Знайди помилку

### Вправа 7

Колега переписав опитування на `asyncio`, але воно й далі триває 2,5 с. Ось його `check_slow`:

```python
async def check_slow(name):
    time.sleep(0.5)
    return f"{name}: відкрито"
```

Знайди причину й напиши виправлене тіло `check_slow` у клітинці нижче, не змінюючи `report_slow`.

In [ ]:
async def check_slow(name):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    await asyncio.sleep(0.5)
    # END SOLUTION
    return f"{name}: відкрито"


async def report_slow():
    return await asyncio.gather(*(check_slow(n) for n in RESTAURANTS))


start = time.perf_counter()
await report_slow()
elapsed = time.perf_counter() - start
print(f"{elapsed:.1f} с")
assert elapsed < 1.5, "Корутини досі виконуються по черзі — шукай блокуючий виклик"
print("✅ Вправа 7 пройдена")

---
## Самоперевірка

1. Чим задача, що чекає, відрізняється від задачі, що рахує? Що прискорює кожну?
2. Чому потоки не прискорили `count_primes`?
3. Як уникнути стану гонитви без `Lock`?
4. Чому `district_stats.orders` у ноутбуці лишився порожнім?
5. Чим `asyncio.run(main())` у файлі відрізняється від `await main()` у ноутбуці?

<details>
<summary>Відповіді</summary>

1. Задача, що чекає, здебільшого простоює (мережа, диск) — допомагають потоки або `asyncio`. Задача, що рахує, займає процесор — допомагають процеси.
2. Через GIL: у кожен момент байт-код виконує лише один потік програми.
3. Не ділити змінних: кожен потік повертає результат, головна програма їх збирає.
4. Процес-працівник має власну копію пам'яті; дані повертаються лише результатами функцій.
5. `asyncio.run` створює цикл подій; у ноутбуці цикл уже працює, тому корутину просто чекають через `await`.

</details>

## Далі

- Теорія, покрокові схеми гонитви й циклу подій, вибір інструменту: [Урок 27](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).
- Документація: [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html), [`threading`](https://docs.python.org/3/library/threading.html), [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html), [`asyncio` — Coroutines and Tasks](https://docs.python.org/3/library/asyncio-task.html).
- **Урок 28** — практикум П6: стек, черга, купа, префіксне дерево й LRU-кеш.